# Learning Spatial Relationships with MISTy on gut spatial data 
**Developed by:** Anna Maguza  
**Affiliation:** Faculty of Medicine, Würzburg University  
**Creation date:** 7th April 2025  
**Last modified date:** 9th April 2025  

In this notebook, we:  
  
* Calculate pathway activities with decoupler  
* Run MISTy to understand how cell surrounding influence on cell behaviour  
* Analyze pathway, transcription factors and CCI influence  

## Import packages

In [1]:
import scanpy as sc
import decoupler as dc
import plotnine as p9
import liana as li
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

import json
from datetime import datetime
import squidpy as sq

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/sin

In [ ]:
from liana.method import MistyData, genericMistyData, lrMistyData
from liana.method.sp import RandomForestModel, LinearModel, RobustLinearModel

## Set up notebooks

In [3]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [4]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (15, 15)

## Load imputed Xenium data

In [5]:
adata = sc.read_h5ad('data/gut_data/maxfuse_checkpoints/gut_hs_XeniumHealthyColon_maxfuse_imputed_and_originalcounts_updated_niches_09042025_131844_log.h5ad')

+ Extract HVGs

In [6]:
sc.pp.highly_variable_genes(
    adata,
    flavor = "seurat",
    n_top_genes = 8000,
    layer = None,
    subset = True,
    span = 1
)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:218: RuntimeWarning: invalid value encountered in log
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:247: RuntimeWarning: invalid value encountered in divide


+ Delete cells where imputation didnt actually work

In [7]:
adata = adata[adata.obs['has_match'] == True].copy()

In [8]:
adata

AnnData object with n_obs × n_vars = 197295 × 8307
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4', 'CD24_ligand_receptor_target_gene_GP', 'SLPI_ligand_receptor_target_gene_GP', 'CXCL14_ligand_receptor_target_gene_GP', 'ANPEP_ligand_receptor_target_gene_GP', 'IL1B_ligand_receptor_target_gene_GP', 'TIMP3_ligand_receptor_target_gene_GP', 'CDH1_ligand_receptor_target_gene_GP', 'TNXB_ligand_receptor_target_gene_GP', 'CLU_ligand_receptor_target_gene_GP', 'TFF1_ligand_receptor_target_gene_GP', 'CCL11_ligand_receptor_target_gene_GP', 'ROBO1_ligand_receptor_target_gene_GP', 'NRG3_ligand_receptor

+ Visualize the dataset with filtered niche

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color=['C_scANVI', 'match_score'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'magma_r',
    ncols=1,
    #crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/xenium_after_imputation.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color=['C_scANVI', 'match_score'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'magma_r',
    ncols=1,
    crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/xenium_after_imputation_zoomed.png", bbox_inches="tight", dpi=300)

## Funconomics

In [11]:
progeny = dc.get_progeny(organism='human', top=500)

python(42080) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [12]:
dc.run_mlm(
    mat=adata,
    net=progeny,
    source='source',
    target='target',
    weight='weight',
    verbose=True,
    use_raw=False,
)

183 features of mat are empty, they will be removed.
Running mlm on mat with 197295 samples and 8124 targets for 14 sources.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [01:00<00:00,  3.00s/it]


In [13]:
acts_progeny = li.ut.obsm_to_adata(adata, 'mlm_estimate')

In [14]:
acts_progeny.var

""
Androgen
EGFR
Estrogen
Hypoxia
JAK-STAT
MAPK
NFkB
PI3K
TGFb
TNFa


In [15]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    acts_progeny,
    library_id="spatial",
    shape=None, 
    color=['JAK-STAT', 'Androgen','Hypoxia', 'MAPK', 'EGFR', 'Estrogen'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'RdBu_r',
    ncols=3,
    #crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/pathways_activity_in_spatial1.png", bbox_inches="tight", dpi=300)

In [16]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    acts_progeny,
    library_id="spatial",
    shape=None, 
    color=['JAK-STAT', 'Androgen','Hypoxia', 'MAPK', 'EGFR', 'Estrogen'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'RdBu_r',
    ncols=3,
    crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/pathways_activity_in_spatial1_zoomed.png", bbox_inches="tight", dpi=300)

In [17]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    acts_progeny,
    library_id="spatial",
    shape=None, 
    color=['TGFb', 'TNFa', 'Trail', 'NFkB', 'PI3K', 'WNT', 'VEGF', 'p53'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'RdBu_r',
    ncols=4,
    #crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/pathways_activity_in_spatial2.png", bbox_inches="tight", dpi=300)

In [18]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

# Your plotting code
sq.pl.spatial_scatter(
    acts_progeny,
    library_id="spatial",
    shape=None, 
    color=['TGFb', 'TNFa', 'Trail', 'NFkB', 'PI3K', 'WNT', 'VEGF', 'p53'],
    size=1,
    frameon=False,
    alpha=1.0,
    cmap = 'RdBu_r',
    ncols=4,
    crop_coord=[(6300, 1950, 7600, 2950)]

)
plt.savefig(f"figures/pathways_activity_in_spatial2_zoomed.png", bbox_inches="tight", dpi=300)

## Formatting & Running MISTy

In [19]:
cell_types = adata.obs['C_scANVI'].unique()
print(f"Found {len(cell_types)} cell types: {cell_types}")

Found 26 cell types: ['Fibroblasts', 'Myofibroblasts', 'Glial cells', 'Macrophages', 'Pericytes', ..., 'BEST4+ epithelial', 'DC', 'Arterial capillary', 'NK', 'Microfold cell']
Length: 26
Categories (26, object): ['B cells', 'BEST4+ epithelial', 'CD4 T', 'CD8 T', ..., 'Stem cells', 'TA', 'Tuft cells', 'Arterial capillary']


In [20]:
compositions = pd.DataFrame(index=adata.obs_names, columns=cell_types)

In [21]:
compositions = compositions.fillna(0)

/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_41364/2262236102.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [22]:
for cell_type in cell_types:
    mask = adata.obs['C_scANVI'] == cell_type
    compositions.loc[mask, cell_type] = 1

In [23]:
adata.obsm['compositions'] = compositions

In [24]:
comps = ad.AnnData(
    X=adata.obsm['compositions'].values,
    obs=adata.obs,
    var=pd.DataFrame(index=adata.obsm['compositions'].columns),
    obsm=adata.obsm,
)

In [25]:
misty = genericMistyData(intra=comps, extra=acts_progeny, cutoff=0.05, bandwidth=200, n_neighs=6)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns fro

## Learn Relationships with MISTy

In [26]:
# Run this if you have time and resources
misty(model=RandomForestModel, n_jobs=-1, verbose = True, seed=1337)

Now learning: Colonocyte: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [04:56<00:00, 148.22s/it]


In [27]:
# Run this if you want it much faster and bit more simplistic 
#misty(model=LinearModel, k_cv=10, seed=1337, bypass_intra=True, verbose = True)

In [28]:
misty.uns['target_metrics'].head()

,target,intra_R2,multi_R2,gain_R2,intra,juxta,para
0,Myofibroblasts,0.073576,0.714378,0.640802,0.0,0.360676,0.639324
1,Colonocyte,0.073529,0.455498,0.381968,0.0,0.406047,0.593953


In [29]:
fig = (li.pl.target_metrics(misty, stat='intra_R2', return_fig=True))
print(fig)
fig.save("figures/misty_intra_R2.png", dpi=300)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_intra_R2.png


<ggplot: (500 x 500)>


In [30]:
fig = (li.pl.target_metrics(misty, stat='gain_R2'))
print(fig)
fig.save("figures/misty_gain_R2.png", dpi=300)

<ggplot: (500 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_gain_R2.png


In [31]:
fig = (li.pl.contributions(misty, return_fig=True))
print(fig)
fig.save("figures/misty_contributions.png", dpi=300)

<ggplot: (500 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_contributions.png


In [32]:
misty.uns['interactions'].head()

,target,predictor,view,importances
0,Myofibroblasts,Colonocyte,intra,1.000000
15,Colonocyte,Myofibroblasts,intra,1.000000
31,Myofibroblasts,Androgen,juxta,0.034762
32,Myofibroblasts,EGFR,juxta,0.026389
33,Myofibroblasts,Estrogen,juxta,0.024682


In [33]:
fig = ((
    li.pl.interactions(misty, view='juxta', return_fig=True, figure_size=(7,5)) +
    p9.scale_fill_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0)
))
print(fig)
fig.save("figures/misty_pathway_importance.png", dpi=300)

<ggplot: (700 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 7 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_pathway_importance.png


## Estimate Transcription Factor activities with decoupler

In [34]:
net = dc.get_collectri()

In [35]:
dc.run_ulm(
    mat=adata,
    net=net,
    verbose=True,
    use_raw=False,
)

183 features of mat are empty, they will be removed.
Running ulm on mat with 197295 samples and 8124 targets for 471 sources.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:36<00:00,  1.84s/it]


In [36]:
acts_tfs = li.ut.obsm_to_adata(adata, 'ulm_estimate')

In [37]:
acts_tfs.var

""
AHR
AIRE
AP1
APEX1
AR
...
ZNF362
ZNF384
ZNF699
ZNF804A


In [38]:
li.ut.spatial_neighbors(acts_tfs, cutoff=0.1, bandwidth=200, set_diag=True)

In [39]:
fig = (li.pl.connectivity(acts_tfs, idx=0, figure_size=(10,10)))
print(fig)
fig.save("figures/misty_liana_connectivity.png", dpi=300)

<ggplot: (1000 x 1000)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 10 x 10 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_liana_connectivity.png


In [40]:
acts_progeny.obsm['spatial'] = acts_tfs.obsm['spatial']
acts_progeny.obsp['spatial_connectivities'] = acts_tfs.obsp['spatial_connectivities']

In [41]:
misty = MistyData(data={"intra": comps, "TFs": acts_tfs, "Pathways": acts_progeny})

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.


In [42]:
misty(model=LinearModel, verbose=True, bypass_intra=True)

Now learning: Fibroblasts:   0%|                                                                                                                                                                                                                                                                                                                     | 0/26 [00:00<?, ?it/s]python(43185) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43186) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43187) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43188) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43190) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43191) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43192) MallocStackLogging: can't turn off

In [43]:
fig = (li.pl.target_metrics(misty, stat='gain_R2'))
print(fig)
fig.save("figures/misty_TF-and_pathway_gain_R2.png", dpi=300)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_TF-and_pathway_gain_R2.png


<ggplot: (500 x 500)>


In [44]:
fig = (li.pl.contributions(misty, return_fig=True))
print(fig)
fig.save("figures/misty_contributions_TFs_and_pathways.png", dpi=300)

<ggplot: (500 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 5 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_contributions_TFs_and_pathways.png


In [45]:
fig = (
li.pl.interactions(misty, view='TFs', top_n=20) +
    p9.labs(x='Transcription Factor', y='Cell type') +
    p9.theme_bw(base_size=14) +
    p9.theme(axis_text_x=p9.element_text(rotation=90, size=13)) +
    # change to blue-red
    p9.scale_fill_gradient2(low='blue', mid='white', high='red')
)

# Display the figure
print(fig)

# Save the figure to a file
fig.save("figures/misty_TFs.png", dpi=300)

<ggplot: (640 x 480)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_TFs.png


## Ligand-Receptor Misty

In [46]:
misty = lrMistyData(adata, bandwidth=200, set_diag=False, cutoff=0.01, nz_threshold=0.1)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns fro

In [47]:
misty(bypass_intra=True, model=LinearModel, verbose=True)

Now learning: NOTCH4:   0%|                                                                                                                                                                                                                                                                                                                         | 0/259 [00:00<?, ?it/s]python(43845) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43846) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43847) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43848) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43849) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43850) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(43852) MallocStackLogging: can't turn off

In [48]:
fig = (
    li.pl.interactions(misty, view='extra', return_fig=True, figure_size=(6, 5), top_n=25, key=abs) +
    p9.scale_fill_gradient2(low="blue", mid="white", high="red", midpoint=0) +
    p9.labs(y='Receptor', x='Ligand')
)

# Display the figure
print(fig)

# Save the figure to a file
fig.save("figures/misty_ligand_receptors_interactions.png", dpi=300)

<ggplot: (600 x 500)>


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6 x 5 in image.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: /Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/6_cci_on_imputed_spatial/figures/misty_ligand_receptors_interactions.png
